# Model Evaluation and Selection

Comprehensive guide to evaluating machine learning models and selecting the best one.

## Overview

This notebook covers:
- Confusion matrix and metrics (precision, recall, F1)
- ROC curves and AUC
- Precision-Recall curves
- Cross-validation strategies
- Learning curves
- Hyperparameter tuning
- Model calibration
- When to use which metric

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GridSearchCV, cross_val_score, learning_curve
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import roc_curve, auc, roc_auc_score, precision_recall_curve, auc as auc_pr
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')


## 1. Create Synthetic Binary Classification Dataset

In [ ]:
# Create synthetic binary classification dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15,
                            n_redundant=5, random_state=42, class_sep=1.0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')
print(f'Class distribution (train): {np.bincount(y_train)}')
print(f'Class distribution (test): {np.bincount(y_test)}')


## 2. Confusion Matrix and Basic Metrics

In [ ]:
# Train a logistic regression model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('Confusion Matrix:')
print(cm)
print(f'\nTrue Negatives: {tn}, False Positives: {fp}')
print(f'False Negatives: {fn}, True Positives: {tp}')


In [ ]:
# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title('Confusion Matrix - Logistic Regression')
plt.tight_layout()
plt.show()


## 3. Precision, Recall, and F1 Score

In [ ]:
# Compute metrics manually
manual_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
manual_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
manual_f1 = 2 * (manual_precision * manual_recall) / (manual_precision + manual_recall) if (manual_precision + manual_recall) > 0 else 0

print(f'Manual Precision: {manual_precision:.4f}')
print(f'Manual Recall: {manual_recall:.4f}')
print(f'Manual F1 Score: {manual_f1:.4f}')

# Compute with sklearn
sk_precision = precision_score(y_test, y_pred)
sk_recall = recall_score(y_test, y_pred)
sk_f1 = f1_score(y_test, y_pred)
sk_accuracy = accuracy_score(y_test, y_pred)

print(f'\nSklearn Precision: {sk_precision:.4f}')
print(f'Sklearn Recall: {sk_recall:.4f}')
print(f'Sklearn F1 Score: {sk_f1:.4f}')
print(f'Sklearn Accuracy: {sk_accuracy:.4f}')


## 4. ROC Curve and AUC

In [ ]:
# Train multiple models for comparison
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict_proba(X_test_scaled)[:, 1]

# Random Forest
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict_proba(X_test)[:, 1]

# SVM
svm = SVC(probability=True, random_state=42, kernel='rbf')
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict_proba(X_test_scaled)[:, 1]

print('Models trained for ROC comparison')


In [ ]:
# Compute ROC curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_lr)
auc_lr = auc(fpr_lr, tpr_lr)

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_rf)
auc_rf = auc(fpr_rf, tpr_rf)

fpr_svm, tpr_svm, _ = roc_curve(y_test, y_pred_svm)
auc_svm = auc(fpr_svm, tpr_svm)

# Plot ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={auc_lr:.3f})', lw=2)
ax.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={auc_rf:.3f})', lw=2)
ax.plot(fpr_svm, tpr_svm, label=f'SVM (AUC={auc_svm:.3f})', lw=2)
ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - Model Comparison')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Precision-Recall Curve

In [ ]:
# Compute precision-recall curves
precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_pred_lr)
auc_pr_lr = auc_pr(recall_lr, precision_lr)

precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_pred_rf)
auc_pr_rf = auc_pr(recall_rf, precision_rf)

precision_svm, recall_svm, _ = precision_recall_curve(y_test, y_pred_svm)
auc_pr_svm = auc_pr(recall_svm, precision_svm)

# Plot precision-recall curves
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall_lr, precision_lr, label=f'Logistic Regression (AUC-PR={auc_pr_lr:.3f})', lw=2)
ax.plot(recall_rf, precision_rf, label=f'Random Forest (AUC-PR={auc_pr_rf:.3f})', lw=2)
ax.plot(recall_svm, precision_svm, label=f'SVM (AUC-PR={auc_pr_svm:.3f})', lw=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves - Model Comparison')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()


## 6. Cross-Validation: K-Fold vs Stratified K-Fold

In [ ]:
# K-Fold Cross-Validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_kfold = cross_val_score(lr, X_train_scaled, y_train, cv=kfold, scoring='f1')

# Stratified K-Fold Cross-Validation
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_skfold = cross_val_score(lr, X_train_scaled, y_train, cv=skfold, scoring='f1')

print('K-Fold CV Scores:', cv_scores_kfold)
print(f'K-Fold Mean: {cv_scores_kfold.mean():.4f} (+/- {cv_scores_kfold.std():.4f})')

print('\nStratified K-Fold CV Scores:', cv_scores_skfold)
print(f'Stratified K-Fold Mean: {cv_scores_skfold.mean():.4f} (+/- {cv_scores_skfold.std():.4f})')


In [ ]:
# Visualize CV comparison
fig, ax = plt.subplots(figsize=(8, 5))
fold_indices = np.arange(1, 6)
ax.plot(fold_indices, cv_scores_kfold, 'o-', label='K-Fold', linewidth=2, markersize=8)
ax.plot(fold_indices, cv_scores_skfold, 's-', label='Stratified K-Fold', linewidth=2, markersize=8)
ax.axhline(y=cv_scores_kfold.mean(), color='C0', linestyle='--', alpha=0.5)
ax.axhline(y=cv_scores_skfold.mean(), color='C1', linestyle='--', alpha=0.5)
ax.set_xlabel('Fold Number')
ax.set_ylabel('F1 Score')
ax.set_title('K-Fold vs Stratified K-Fold Cross-Validation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Learning Curves

In [ ]:
# Plot learning curves
train_sizes, train_scores, val_scores = learning_curve(
    LogisticRegression(random_state=42, max_iter=1000),
    X_train_scaled, y_train,
    cv=5, scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(train_sizes, train_mean, 'o-', label='Training Score', linewidth=2)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2)
ax.plot(train_sizes, val_mean, 's-', label='Validation Score', linewidth=2)
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2)
ax.set_xlabel('Training Set Size')
ax.set_ylabel('F1 Score')
ax.set_title('Learning Curve - Logistic Regression')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Hyperparameter Tuning with GridSearchCV

In [ ]:
# GridSearchCV for Random Forest
param_grid = {
    'n_estimators': [10, 30, 50],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5]
}

rf_base = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf_base, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV F1 Score: {grid_search.best_score_:.4f}')

# Test set performance
best_rf = grid_search.best_estimator_
test_f1 = f1_score(y_test, best_rf.predict(X_test))
print(f'Test Set F1 Score: {test_f1:.4f}')


## 9. Model Calibration

In [ ]:
# Compute calibration curves
prob_true_lr, prob_pred_lr = calibration_curve(y_test, y_pred_lr, n_bins=10)
prob_true_rf, prob_pred_rf = calibration_curve(y_test, y_pred_rf, n_bins=10)
prob_true_svm, prob_pred_svm = calibration_curve(y_test, y_pred_svm, n_bins=10)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
ax.plot(prob_pred_lr, prob_true_lr, 'o-', label='Logistic Regression', linewidth=2)
ax.plot(prob_pred_rf, prob_true_rf, 's-', label='Random Forest', linewidth=2)
ax.plot(prob_pred_svm, prob_true_svm, '^-', label='SVM', linewidth=2)
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curves')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()


## 10. Metric Comparison and Selection

In [ ]:
# Create a comparison table
metrics_comparison = {
    'Model': ['Logistic Regression', 'Random Forest', 'SVM'],
    'Accuracy': [
        accuracy_score(y_test, (y_pred_lr > 0.5).astype(int)),
        accuracy_score(y_test, rf.predict(X_test)),
        accuracy_score(y_test, (y_pred_svm > 0.5).astype(int))
    ],
    'Precision': [
        precision_score(y_test, (y_pred_lr > 0.5).astype(int)),
        precision_score(y_test, rf.predict(X_test)),
        precision_score(y_test, (y_pred_svm > 0.5).astype(int))
    ],
    'Recall': [
        recall_score(y_test, (y_pred_lr > 0.5).astype(int)),
        recall_score(y_test, rf.predict(X_test)),
        recall_score(y_test, (y_pred_svm > 0.5).astype(int))
    ],
    'F1': [
        f1_score(y_test, (y_pred_lr > 0.5).astype(int)),
        f1_score(y_test, rf.predict(X_test)),
        f1_score(y_test, (y_pred_svm > 0.5).astype(int))
    ],
    'AUC-ROC': [auc_lr, auc_rf, auc_svm]
}

import pandas as pd
df_metrics = pd.DataFrame(metrics_comparison)
print(df_metrics.to_string(index=False))


## 11. When to Use Which Metric

- **Accuracy**: When classes are balanced and all errors have equal cost
- **Precision**: When false positives are costly (e.g., spam detection)
- **Recall**: When false negatives are costly (e.g., disease diagnosis)
- **F1 Score**: When balancing precision and recall is important
- **AUC-ROC**: For ranking and comparing models across thresholds
- **AUC-PR**: For imbalanced datasets, especially with rare positive class

## Interview Takeaways

1. **Confusion Matrix Foundation**: Understand TP, TN, FP, FN to compute all metrics
2. **Metric Selection**: Choose metrics aligned with business goals, not just accuracy
3. **ROC vs PR Curves**: Use PR curves for imbalanced data, ROC for balanced
4. **Cross-Validation**: Always stratify for classification to maintain class balance
5. **Learning Curves**: Diagnose bias/variance problems and data needs
6. **Hyperparameter Tuning**: GridSearchCV/RandomSearchCV with cross-validation
7. **Calibration Matters**: Well-calibrated probabilities enable better decision-making
8. **Train/Test Split**: Never tune hyperparameters on test data